# 07_phase3_LIANA_interactions.ipynb
Phase 3 — Cell-Cell Communication Analysis

**Design:** LIANA run per condition, not just once per dataset, so results connect directly to the DE/pathway story already established:
- GSE114725: Tumour vs Normal (2 networks)
- GSE176078: ER+ / HER2+ / TNBC (3 networks)

**Input data:** `adata.raw` (log-normalised, unscaled, full gene set) — the correct input for LIANA, same convention used in Phase 2's marker validation. NOT the scaled/HVG-reduced clustering matrix, and NOT the raw integer counts used for pseudobulk DE.

In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_liana"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_liana"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print(f"LIANA version: {li.__version__}")

Setup complete
LIANA version: 1.7.3


In [2]:
# ----------------------------
# Cell 2 — Load Phase 2 annotated data, extract .raw (log-normalised)
# ----------------------------
def load_for_liana(annotated_path, dataset_name):
    adata_full = sc.read_h5ad(annotated_path)
    adata_liana = adata_full.raw.to_adata()
    adata_liana.obs = adata_full.obs.copy()
    print(f"{dataset_name}: {adata_liana.n_obs} cells x {adata_liana.n_vars} genes "
          f"(log-normalised, full gene set)")
    del adata_full
    gc.collect()
    return adata_liana

adata1_liana = load_for_liana(
    PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad", "GSE114725")
adata2_liana = load_for_liana(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", "GSE176078")

GSE114725: 44662 cells x 14800 genes (log-normalised, full gene set)
GSE176078: 91425 cells x 27343 genes (log-normalised, full gene set)


In [3]:
# ----------------------------
# Cell 3 — LIANA runner function
# Uses rank_aggregate (LIANA's consensus method, combining multiple
# ligand-receptor scoring methods including CellPhoneDB-style analysis)
# for a more robust result than any single method alone.
# Excludes very small cell types (<50 cells) and the known artefact
# cluster from the interaction analysis, same filtering discipline
# applied throughout Phase 2/3.
# ----------------------------
EXCLUDE_FROM_LIANA = ["Unassigned (n=28, doublet/mixed-identity artefact)"]
MIN_CELLS_FOR_LIANA = 50

def run_liana(adata, subset_mask, label, min_cells=MIN_CELLS_FOR_LIANA):
    adata_sub = adata[subset_mask].copy()
    adata_sub = adata_sub[~adata_sub.obs["cell_type"].isin(EXCLUDE_FROM_LIANA)].copy()

    cell_counts = adata_sub.obs["cell_type"].value_counts()
    viable_types = cell_counts[cell_counts >= min_cells].index.tolist()
    excluded_types = cell_counts[cell_counts < min_cells].index.tolist()
    if excluded_types:
        print(f"  Excluding from {label} (< {min_cells} cells): {excluded_types}")

    adata_sub = adata_sub[adata_sub.obs["cell_type"].isin(viable_types)].copy()
    adata_sub.obs["cell_type"] = adata_sub.obs["cell_type"].astype(str)

    print(f"  Running LIANA on {label}: {adata_sub.n_obs} cells, {adata_sub.obs['cell_type'].nunique()} cell types")

    li.mt.rank_aggregate(
        adata_sub,
        groupby="cell_type",
        expr_prop=0.1,
        verbose=False,
        use_raw=False,
    )

    results = adata_sub.uns["liana_res"].copy()
    del adata_sub
    gc.collect()
    return results

print("LIANA runner ready")

LIANA runner ready


In [4]:
# ----------------------------
# Cell 4 — GSE114725: Tumour vs Normal LIANA networks
# ----------------------------
all_liana_1 = {}

for tissue in ["TUMOR", "NORMAL"]:
    mask = (adata1_liana.obs["tissue"] == tissue).values
    n_cells = mask.sum()
    if n_cells < 100:
        print(f"SKIP GSE114725 {tissue}: only {n_cells} cells")
        continue
    results = run_liana(adata1_liana, mask, f"GSE114725_{tissue}")
    all_liana_1[tissue] = results
    results.to_csv(RESULTS_DIR / f"GSE114725_liana_{tissue}.csv", index=False)
    gc.collect()

print("\nGSE114725 LIANA complete")

  Running LIANA on GSE114725_TUMOR: 19594 cells, 9 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Excluding from GSE114725_NORMAL (< 50 cells): ['B cells', 'Mixed/stromal-contaminated (CD8+fibroblast signal)', 'Mast cells', 'pDC']
  Running LIANA on GSE114725_NORMAL: 4239 cells, 5 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual


GSE114725 LIANA complete


In [5]:
# ----------------------------
# Cell 5 — GSE176078: per-subtype LIANA networks
# ----------------------------
all_liana_2 = {}

for subtype in ["ER+", "HER2+", "TNBC"]:
    mask = (adata2_liana.obs["subtype"] == subtype).values
    n_cells = mask.sum()
    if n_cells < 100:
        print(f"SKIP GSE176078 {subtype}: only {n_cells} cells")
        continue
    safe_subtype = subtype.replace("+", "plus")
    results = run_liana(adata2_liana, mask, f"GSE176078_{subtype}")
    all_liana_2[subtype] = results
    results.to_csv(RESULTS_DIR / f"GSE176078_liana_{safe_subtype}.csv", index=False)
    gc.collect()

print("\nGSE176078 LIANA complete")

  Running LIANA on GSE176078_ER+: 33552 cells, 16 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Running LIANA on GSE176078_HER2+: 18364 cells, 16 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Running LIANA on GSE176078_TNBC: 39481 cells, 16 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual


GSE176078 LIANA complete


In [7]:
# ----------------------------
# Cell 6 — Identify top, high-confidence interactions per network
# FIX: sorting by magnitude_rank alone surfaced the same ubiquitous
# interaction (B2M -> KLRD1, MHC-I sensing by NK cells) repeated across
# nearly every row, just with a different source cell type each time —
# B2M is near-universally expressed, so it mathematically dominates any
# magnitude-sorted list regardless of biological distinctiveness.
# Switched to specificity_rank (how unusually strong an interaction is
# for THIS particular cell-type pair, not just how highly expressed the
# genes are overall) and de-duplicated by ligand-receptor pair, so the
# top list shows a varied set of distinctive interactions rather than
# repeats of one dominant, non-specific signal.
# ----------------------------
def top_interactions(results_df, n=15, dedupe_by_pair=True):
    df = results_df.sort_values("specificity_rank").copy()
    if dedupe_by_pair:
        df = df.drop_duplicates(subset=["ligand_complex", "receptor_complex"], keep="first")
    return df.head(n)[
        ["source", "target", "ligand_complex", "receptor_complex", "magnitude_rank", "specificity_rank"]
    ]

print("=== GSE114725 top interactions (by specificity_rank) ===")
for tissue, results in all_liana_1.items():
    print(f"\n{tissue}:")
    top = top_interactions(results, n=10)
    print(top.to_string(index=False))
    top.to_csv(RESULTS_DIR / f"GSE114725_liana_{tissue}_top10.csv", index=False)

print("\n\n=== GSE176078 top interactions (by specificity_rank) ===")
for subtype, results in all_liana_2.items():
    print(f"\n{subtype}:")
    top = top_interactions(results, n=10)
    print(top.to_string(index=False))
    safe_subtype = subtype.replace("+", "plus")
    top.to_csv(RESULTS_DIR / f"GSE176078_liana_{safe_subtype}_top10.csv", index=False)

=== GSE114725 top interactions (by specificity_rank) ===

TUMOR:
      source                                             target ligand_complex receptor_complex  magnitude_rank  specificity_rank
Monocytes/DC                                       Monocytes/DC          CXCL8            CXCR2        0.037556      1.270425e-08
Monocytes/DC                                       Monocytes/DC          CXCL8            CXCR1        0.118833      5.878873e-08
Monocytes/DC                                            B cells          CXCL8            CD79A        0.008237      7.823872e-08
  Mast cells                                       Monocytes/DC           CSF1            CSF3R        0.034038      2.406312e-07
 Macrophages                                            B cells            FN1            CD79A        0.022428      1.583616e-06
Monocytes/DC Mixed/stromal-contaminated (CD8+fibroblast signal)          CXCL8            ACKR1        0.229737      3.475566e-06
 Macrophages             

In [8]:
# Sanity check — SCT (secretin) expression in pDC, before featuring this
# prominently. LIANA's specificity_rank can be skewed by a small number
# of outlier cells with unusually high expression, especially in a
# already-rare population (pDC).
from scipy.sparse import issparse

for subtype in ["ER+", "HER2+", "TNBC"]:
    mask = (adata2_liana.obs["subtype"] == subtype) & (adata2_liana.obs["cell_type"] == "pDC")
    n_pdc = mask.sum()
    if "SCT" in adata2_liana.var_names:
        X = adata2_liana[mask.values, "SCT"].X
        if issparse(X): X = X.toarray()
        pct_expressing = (X > 0).mean() * 100
        mean_expr = X.mean()
        print(f"{subtype}: {n_pdc} pDC cells, SCT expressed in {pct_expressing:.1f}% of them, "
              f"mean expression = {mean_expr:.3f}")
    else:
        print(f"{subtype}: SCT not found in var_names")

print(f"\nTotal pDC across dataset: {(adata2_liana.obs['cell_type'] == 'pDC').sum()}")
print(f"(For reference: pDC was {315} cells total out of 91,425 in Phase 2 — one of the rarest cell types)")

ER+: 91 pDC cells, SCT expressed in 60.4% of them, mean expression = 1.061
HER2+: 68 pDC cells, SCT expressed in 69.1% of them, mean expression = 1.167
TNBC: 156 pDC cells, SCT expressed in 62.8% of them, mean expression = 1.140

Total pDC across dataset: 315
(For reference: pDC was 315 cells total out of 91,425 in Phase 2 — one of the rarest cell types)


In [10]:
# ----------------------------
# Cell 7 — LIANA dot plots per network
# FIX: panel titles were overlapping/cut off — too many source cell type
# panels crammed into a fixed 12-inch width. Figure width now scales with
# the number of cell types, and top_n reduced from 15 to 10 interactions
# (closer to thesis-figure scope anyway, and naturally reduces crowding).
# ----------------------------
import liana as li

def plot_liana_dotplot(adata, subset_mask, label, save_path, top_n=10):
    adata_sub = adata[subset_mask].copy()
    adata_sub = adata_sub[~adata_sub.obs["cell_type"].isin(EXCLUDE_FROM_LIANA)].copy()
    cell_counts = adata_sub.obs["cell_type"].value_counts()
    viable_types = cell_counts[cell_counts >= MIN_CELLS_FOR_LIANA].index.tolist()
    adata_sub = adata_sub[adata_sub.obs["cell_type"].isin(viable_types)].copy()
    adata_sub.obs["cell_type"] = adata_sub.obs["cell_type"].astype(str)

    li.mt.rank_aggregate(adata_sub, groupby="cell_type", expr_prop=0.1, verbose=False, use_raw=False)

    n_types = adata_sub.obs["cell_type"].nunique()
    fig = li.pl.dotplot(
        adata=adata_sub,
        colour="magnitude_rank",
        size="specificity_rank",
        top_n=top_n,
        orderby="specificity_rank",
        orderby_ascending=True,
        target_labels=None,
        figure_size=(max(14, n_types * 2.2), 8),
    )
    fig.save(save_path, dpi=300, width=max(14, n_types * 2.2), height=8, limitsize=False)
    print(f"  Saved {save_path.name}")

    del adata_sub
    gc.collect()

for tissue in all_liana_1.keys():
    mask = (adata1_liana.obs["tissue"] == tissue).values
    plot_liana_dotplot(adata1_liana, mask, f"GSE114725_{tissue}",
                        FIGURE_DIR / f"GSE114725_liana_dotplot_{tissue}.png")

for subtype in all_liana_2.keys():
    mask = (adata2_liana.obs["subtype"] == subtype).values
    safe_subtype = subtype.replace("+", "plus")
    plot_liana_dotplot(adata2_liana, mask, f"GSE176078_{subtype}",
                        FIGURE_DIR / f"GSE176078_liana_dotplot_{safe_subtype}.png")

print("\nAll LIANA dot plots saved")

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Saved GSE114725_liana_dotplot_TUMOR.png


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Saved GSE114725_liana_dotplot_NORMAL.png


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Saved GSE176078_liana_dotplot_ERplus.png


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Saved GSE176078_liana_dotplot_HER2plus.png


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual

  Saved GSE176078_liana_dotplot_TNBC.png

All LIANA dot plots saved
